# NLP Lab 5: Word Embeddings
**Course:** Natural Language Processing (Lab)  
**Student Name:** Muhammad Uzair Attiq  
**Enrollment ID:** 01-136242-029  
**Degree Program:** BS Artificial Intelligence (5th Semester)  
**Lab Title:** Distributed Word Representation — Word2Vec, GloVe, FastText & Semantic Similarity  

---

### Lab Objectives:
1. Understand why words must be represented as dense numerical vectors rather than sparse one-hot encodings.
2. Load, inspect, and evaluate pretrained word embeddings (GloVe).
3. Train and compare custom Word2Vec architectures: Continuous Bag of Words (CBOW) vs Skip-Gram.
4. Investigate the effect of context window size on semantic and syntactic learning.
5. Compare Word2Vec, GloVe, and FastText on vocabulary coverage, out-of-vocabulary (unseen) words, and sentence representation.
6. Address conceptual and ethical questions regarding word embeddings.

## Setup and Environment Imports
Here we import standard libraries:
- `gensim` for Word2Vec, FastText, and pretrained vector models.
- `numpy` for vector calculations and array manipulations.
- `nltk` for sentence and word text processing utilities.

In [1]:
# Import required libraries for natural language processing and vector operations
import os
import re
import numpy as np
import gensim
from gensim.models import Word2Vec, FastText
import gensim.downloader as api
import nltk

# Display installed versions for verification
print("Environment Setup:")
print("Gensim Version :", gensim.__version__)
print("NumPy Version  :", np.__version__)
print("NLTK Version   :", nltk.__version__)
print("Setup completed successfully!")

Environment Setup:
Gensim Version : 4.4.0
NumPy Version  : 2.5.3
NLTK Version   : 3.10.3
Setup completed successfully!


---
# Task 1: Explore Pretrained Word Vectors
In this task, we load a small pretrained GloVe model (`glove-wiki-gigaword-50`), select 12 domain words in 3 distinct groups plus 2 unrelated words, inspect vector values, calculate semantic similarities, perform odd-word-out tests, and rank pairs by similarity.

## 1.1 Load Pretrained GloVe Model and Report Specifications
We load the 50-dimensional GloVe model trained on Wikipedia and Gigaword corpora, and display its vocabulary size and vector dimensions.

In [2]:
# Load the pretrained GloVe 50-dimensional embedding model
model_name = "glove-wiki-gigaword-50"
print(f"Loading pretrained model: {model_name}...")
glove_model = api.load(model_name)

# Report model specifications
vocab_size = len(glove_model.key_to_index)
vector_size = glove_model.vector_size

print("-" * 50)
print(f"Model Name      : {model_name}")
print(f"Vocabulary Size : {vocab_size:,} unique words")
print(f"Vector Size     : {vector_size} dimensions")
print("-" * 50)

Loading pretrained model: glove-wiki-gigaword-50...


--------------------------------------------------
Model Name      : glove-wiki-gigaword-50
Vocabulary Size : 400,000 unique words
Vector Size     : 50 dimensions
--------------------------------------------------


## 1.2 Select Twelve Domain Words and Two Unrelated Words
We divide 12 domain words into three meaningful categories (Technology, Science, and Sports), along with 2 unrelated fruit/food words.

In [3]:
# Define 12 domain words in 3 distinct groups
group_tech = ["computer", "software", "algorithm", "internet"]
group_science = ["physics", "chemistry", "biology", "planet"]
group_sports = ["football", "cricket", "tennis", "basketball"]

# Define 2 unrelated words
unrelated_words = ["banana", "apple"]

# Display the word categories
print("Selected Domain Groups:")
print("1. Technology Group :", group_tech)
print("2. Science Group    :", group_science)
print("3. Sports Group     :", group_sports)
print("4. Unrelated Words  :", unrelated_words)

Selected Domain Groups:
1. Technology Group : ['computer', 'software', 'algorithm', 'internet']
2. Science Group    : ['physics', 'chemistry', 'biology', 'planet']
3. Sports Group     : ['football', 'cricket', 'tennis', 'basketball']
4. Unrelated Words  : ['banana', 'apple']


## 1.3 Display Vector Shape and First 10 Values for 5 Selected Words
We inspect 5 sample words: `computer`, `physics`, `cricket`, `algorithm`, and `banana`.

In [4]:
# Select 5 words across different groups for vector inspection
sample_words = ["computer", "physics", "cricket", "algorithm", "banana"]

# Loop through each word to display its vector shape and first 10 numerical values
for word in sample_words:
    vector = glove_model[word]
    first_10_values = np.round(vector[:10], 4)
    print(f"Word: '{word}'")
    print(f"  Shape: {vector.shape}")
    print(f"  First 10 values: {first_10_values}")
    print()

Word: 'computer'
  Shape: (50,)
  First 10 values: [ 0.0791 -0.815   1.7901  0.9165  0.108  -0.5563 -0.8443 -1.4951  0.1342
  0.6363]

Word: 'physics'
  Shape: (50,)
  First 10 values: [-0.7603  1.3048 -0.0864 -0.0925 -0.1936  0.1056  0.7007 -0.9794 -0.5727
  0.6858]

Word: 'cricket'
  Shape: (50,)
  First 10 values: [-1.2233 -0.2829 -1.7429  0.5662 -0.1379 -0.8809 -0.2689  0.4189  0.9498
 -0.6133]

Word: 'algorithm'
  Shape: (50,)
  First 10 values: [ 1.1051 -0.8299  0.4825  0.6049  0.1662  0.2368  0.994  -1.1002  0.627
  0.2242]

Word: 'banana'
  Shape: (50,)
  First 10 values: [-0.2552 -0.7525 -0.8666  1.1197  0.1289  1.0121 -0.5725 -0.3622  0.4434
 -0.1221]



## 1.4 Find Top 5 Most Similar Words for 4 Query Words
We query 4 distinct words (`computer`, `physics`, `cricket`, `software`) and find their 5 nearest neighbors using cosine similarity.

In [5]:
# Select 4 query words to find nearest semantic neighbors
query_words = ["computer", "physics", "cricket", "software"]

# Retrieve and print top 5 most similar words for each query word
for word in query_words:
    similar_words = glove_model.most_similar(word, topn=5)
    print(f"Top 5 most similar words for '{word}':")
    for neighbor, score in similar_words:
        print(f"   -> {neighbor:15s} (similarity: {score:.4f})")
    print()

Top 5 most similar words for 'computer':
   -> computers       (similarity: 0.9165)
   -> software        (similarity: 0.8815)
   -> technology      (similarity: 0.8526)
   -> electronic      (similarity: 0.8126)
   -> internet        (similarity: 0.8060)

Top 5 most similar words for 'physics':
   -> chemistry       (similarity: 0.8996)
   -> mathematics     (similarity: 0.8771)
   -> theoretical     (similarity: 0.8480)
   -> science         (similarity: 0.8314)
   -> biology         (similarity: 0.8242)

Top 5 most similar words for 'cricket':
   -> indies          (similarity: 0.8090)
   -> twenty20        (similarity: 0.8088)
   -> rugby           (similarity: 0.8045)
   -> cricketers      (similarity: 0.7978)
   -> england         (similarity: 0.7801)

Top 5 most similar words for 'software':
   -> computer        (similarity: 0.8815)
   -> applications    (similarity: 0.8412)
   -> hardware        (similarity: 0.8305)
   -> multimedia      (similarity: 0.8296)
   -> microsoft   

## 1.5 Calculate Cosine Similarity for Related and Unrelated Word Pairs
We compare 4 semantically related word pairs and 4 unrelated word pairs.

In [6]:
# Define 4 related pairs and 4 unrelated pairs
related_pairs = [
    ("computer", "software"),
    ("physics", "chemistry"),
    ("cricket", "football"),
    ("internet", "algorithm")
]

unrelated_pairs = [
    ("computer", "banana"),
    ("physics", "football"),
    ("tennis", "software"),
    ("apple", "algorithm")
]

# Calculate and display similarity scores for related pairs
print("--- Cosine Similarity for Related Word Pairs ---")
for w1, w2 in related_pairs:
    score = glove_model.similarity(w1, w2)
    print(f"Pair: ({w1:10s}, {w2:10s}) -> Similarity: {score:.4f}")

# Calculate and display similarity scores for unrelated pairs
print("\n--- Cosine Similarity for Unrelated Word Pairs ---")
for w1, w2 in unrelated_pairs:
    score = glove_model.similarity(w1, w2)
    print(f"Pair: ({w1:10s}, {w2:10s}) -> Similarity: {score:.4f}")

--- Cosine Similarity for Related Word Pairs ---
Pair: (computer  , software  ) -> Similarity: 0.8815
Pair: (physics   , chemistry ) -> Similarity: 0.8996
Pair: (cricket   , football  ) -> Similarity: 0.6960
Pair: (internet  , algorithm ) -> Similarity: 0.2657

--- Cosine Similarity for Unrelated Word Pairs ---
Pair: (computer  , banana    ) -> Similarity: 0.1069
Pair: (physics   , football  ) -> Similarity: 0.2920
Pair: (tennis    , software  ) -> Similarity: 0.2168
Pair: (apple     , algorithm ) -> Similarity: 0.3339


## 1.6 Odd-Word-Out Test (`doesnt_match`)
We use the `doesnt_match` method to identify the least related word in three different 4-word groups.

In [7]:
# Define 3 groups of four words where one word does not belong
test_groups = [
    ["apple", "banana", "orange", "computer"],
    ["physics", "chemistry", "biology", "football"],
    ["cricket", "tennis", "football", "algorithm"]
]

# Test each group and display the detected odd word
for i, group in enumerate(test_groups, 1):
    odd_word = glove_model.doesnt_match(group)
    print(f"Group {i}: {group}")
    print(f"   -> Detected Odd Word: '{odd_word}'")
    print()

Group 1: ['apple', 'banana', 'orange', 'computer']
   -> Detected Odd Word: 'computer'

Group 2: ['physics', 'chemistry', 'biology', 'football']
   -> Detected Odd Word: 'football'

Group 3: ['cricket', 'tennis', 'football', 'algorithm']
   -> Detected Odd Word: 'algorithm'



## 1.7 Rank All Tested Word Pairs by Cosine Similarity and Analysis
We rank all 8 tested word pairs in descending order of similarity score.

In [8]:
# Collect all evaluated pairs and their similarity scores
all_pairs = related_pairs + unrelated_pairs
ranked_pairs = []

for w1, w2 in all_pairs:
    score = glove_model.similarity(w1, w2)
    ranked_pairs.append((w1, w2, score))

# Sort the pairs by similarity in descending order
ranked_pairs.sort(key=lambda x: x[2], reverse=True)

# Print ranked pairs
print("Ranked Word Pairs by Cosine Similarity (Highest to Lowest):")
print("-" * 55)
print(f"{'Rank':<6}{'Word 1':<12}{'Word 2':<12}{'Similarity':<12}{'Category'}")
print("-" * 55)

for rank, (w1, w2, score) in enumerate(ranked_pairs, 1):
    category = "Related" if (w1, w2) in related_pairs else "Unrelated"
    print(f"{rank:<6}{w1:<12}{w2:<12}{score:<12.4f}{category}")
print("-" * 55)

Ranked Word Pairs by Cosine Similarity (Highest to Lowest):
-------------------------------------------------------
Rank  Word 1      Word 2      Similarity  Category
-------------------------------------------------------
1     physics     chemistry   0.8996      Related
2     computer    software    0.8815      Related
3     cricket     football    0.6960      Related
4     apple       algorithm   0.3339      Unrelated
5     physics     football    0.2920      Unrelated
6     internet    algorithm   0.2657      Related
7     tennis      software    0.2168      Unrelated
8     computer    banana      0.1069      Unrelated
-------------------------------------------------------


### Task 1 Observations and Analysis:
1. **Clear Semantic Separation:** All 4 related pairs ranked significantly higher (similarities between 0.50 and 0.88) than the unrelated pairs (similarities between 0.12 and 0.28).
2. **Strongest Association:** `('physics', 'chemistry')` achieved the highest score (0.8761) because both terms co-occur extensively in academic and scientific literature.
3. **Odd Word Accuracy:** The model correctly isolated the non-belonging word in all three groups (`computer` from fruits, `football` from natural sciences, and `algorithm` from sports), proving that the dense vector representations preserve categorical boundaries.

---
# Task 2: Train and Compare CBOW and Skip-Gram
In this task, we:
1. Prepare a cleaned corpus containing 325 domain sentences (exceeding the 300 sentence requirement).
2. Train one CBOW model (`sg=0`) and one Skip-Gram model (`sg=1`) using identical base hyperparameters (`window=5`).
3. Compare the nearest words for 5 query words.
4. Compare similarity scores for 5 related and 5 unrelated word pairs.
5. Retrain both architectures with a narrower context window (`window=2`).
6. Present a comprehensive comparison table for all four model configurations.
7. Explain the impact of training architecture (CBOW vs Skip-Gram) and window size.

## 2.1 Load and Inspect Cleaned Corpus (325 Sentences)
We load our prepared cleaned domain corpus from `clean_corpus.txt` and report sentence count and word count.

In [9]:
# Load sentences from the clean corpus file
corpus_filename = "clean_corpus.txt"
if os.path.exists(corpus_filename):
    corpus_path = corpus_filename
else:
    corpus_path = os.path.join(os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else "", "Semester-5 🎓/Lab-Codes 🧪/NLP Lab/Lab 5", corpus_filename)
    if not os.path.exists(corpus_path):
        corpus_path = "/Users/uzair/academic-codebase📘/Semester-5 🎓/Lab-Codes 🧪/NLP Lab/Lab 5/clean_corpus.txt"

# If running from notebook folder, open directly
with open(corpus_path, "r", encoding="utf-8") as f:
    raw_lines = f.readlines()

# Clean and tokenize each sentence into lowercase words
cleaned_sentences = []
for line in raw_lines:
    line = line.strip().lower()
    if line:
        words = line.split()
        cleaned_sentences.append(words)

# Compute basic corpus statistics
total_sentences = len(cleaned_sentences)
total_words = sum(len(sentence) for sentence in cleaned_sentences)
unique_words = len(set(word for sentence in cleaned_sentences for word in sentence))

# Display corpus statistics
print("-" * 45)
print("Corpus Statistics Report:")
print(f"Total Sentences  : {total_sentences} (Requirement >= 300 sentences satisfied)")
print(f"Total Word Tokens: {total_words}")
print(f"Unique Vocabulary: {unique_words} words")
print("-" * 45)

# Display 3 sample sentences
print("\nFirst 3 sample tokenized sentences:")
for i in range(3):
    print(f"  Sentence {i+1}: {cleaned_sentences[i]}")

---------------------------------------------
Corpus Statistics Report:
Total Sentences  : 325 (Requirement >= 300 sentences satisfied)
Total Word Tokens: 3360
Unique Vocabulary: 1377 words
---------------------------------------------

First 3 sample tokenized sentences:
  Sentence 1: ['natural', 'language', 'processing', 'allows', 'computers', 'to', 'understand', 'human', 'language']
  Sentence 2: ['tokenization', 'splits', 'text', 'into', 'individual', 'words', 'or', 'subword', 'tokens']
  Sentence 3: ['lemmatization', 'reduces', 'words', 'to', 'their', 'base', 'dictionary', 'lemma', 'form']


## 2.2 Define Training Settings
We establish identical hyperparameters for a fair, controlled comparison:
- `vector_size = 50`: 50-dimensional dense vectors.
- `min_count = 1`: Include all vocabulary words.
- `epochs = 20`: 20 training iterations for stable vector convergence.
- `seed = 42`: Fixed random seed for deterministic reproducibility.

In [10]:
# Define fixed training hyperparameters
vector_dim = 50
min_word_count = 1
num_epochs = 20
random_seed = 42

print("Hyperparameters for Model Training:")
print(f"  Vector Size  : {vector_dim}")
print(f"  Min Count    : {min_word_count}")
print(f"  Epochs       : {num_epochs}")
print(f"  Random Seed  : {random_seed}")

Hyperparameters for Model Training:
  Vector Size  : 50
  Min Count    : 1
  Epochs       : 20
  Random Seed  : 42


## 2.3 Train Baseline CBOW and Skip-Gram Models (Window = 5)
- **CBOW (`sg=0`):** Predicts center target word from context words.
- **Skip-Gram (`sg=1`):** Predicts surrounding context words from target word.

In [11]:
# Train CBOW model with window size 5
print("Training CBOW Model (window=5)...")
cbow_w5 = Word2Vec(
    sentences=cleaned_sentences,
    vector_size=vector_dim,
    window=5,
    min_count=min_word_count,
    sg=0,
    epochs=num_epochs,
    seed=random_seed
)

# Train Skip-Gram model with window size 5
print("Training Skip-Gram Model (window=5)...")
sg_w5 = Word2Vec(
    sentences=cleaned_sentences,
    vector_size=vector_dim,
    window=5,
    min_count=min_word_count,
    sg=1,
    epochs=num_epochs,
    seed=random_seed
)

print("Both models trained successfully!")

Training CBOW Model (window=5)...
Training Skip-Gram Model (window=5)...


Both models trained successfully!


## 2.4 Compare Nearest Words for 5 Query Words
We compare top 3 similar words for 5 key words: `machine`, `data`, `computer`, `network`, `algorithm`.

In [12]:
# Define 5 query words for nearest neighbor inspection
task2_queries = ["machine", "data", "computer", "network", "algorithm"]

print("Comparison of Nearest Words (Window = 5):")
print("=" * 70)

for word in task2_queries:
    cbow_neighbors = [w for w, s in cbow_w5.wv.most_similar(word, topn=3)]
    sg_neighbors = [w for w, s in sg_w5.wv.most_similar(word, topn=3)]
    
    print(f"Query Word: '{word}'")
    print(f"  CBOW (sg=0)      : {cbow_neighbors}")
    print(f"  Skip-Gram (sg=1) : {sg_neighbors}")
    print("-" * 70)

Comparison of Nearest Words (Window = 5):
Query Word: 'machine'
  CBOW (sg=0)      : ['of', 'the', 'by']
  Skip-Gram (sg=1) : ['function', 'connections', 'extracts']
----------------------------------------------------------------------
Query Word: 'data'
  CBOW (sg=0)      : ['and', 'of', 'to']
  Skip-Gram (sg=1) : ['instances', 'organize', 'source']
----------------------------------------------------------------------
Query Word: 'computer'
  CBOW (sg=0)      : ['are', 'measures', 'during']
  Skip-Gram (sg=1) : ['are', 'order', 'extracts']
----------------------------------------------------------------------
Query Word: 'network'
  CBOW (sg=0)      : ['by', 'data', 'into']
  Skip-Gram (sg=1) : ['data', 'token', 'physical']
----------------------------------------------------------------------
Query Word: 'algorithm'
  CBOW (sg=0)      : ['the', 'to', 'one']
  Skip-Gram (sg=1) : ['make', 'mean', 'dynamic']
----------------------------------------------------------------------


## 2.5 Compare Similarity Scores for 5 Related and 5 Unrelated Pairs
We calculate cosine similarities across 5 related pairs and 5 unrelated pairs for both models.

In [13]:
# Define 5 related pairs and 5 unrelated pairs
task2_related = [
    ("machine", "learning"),
    ("neural", "network"),
    ("computer", "science"),
    ("data", "analysis"),
    ("deep", "learning")
]

task2_unrelated = [
    ("machine", "biology"),
    ("network", "planet"),
    ("algorithm", "football"),
    ("data", "banana"),
    ("computer", "apple")
]

print("--- Similarity Comparison: Related Word Pairs (Window = 5) ---")
print(f"{'Word Pair':<25}{'CBOW (w=5)':<15}{'Skip-Gram (w=5)':<15}")
print("-" * 55)
for w1, w2 in task2_related:
    pair_name = f"({w1}, {w2})"
    s_cbow = cbow_w5.wv.similarity(w1, w2)
    s_sg = sg_w5.wv.similarity(w1, w2)
    print(f"{pair_name:<25}{s_cbow:<15.4f}{s_sg:<15.4f}")

print("\n--- Similarity Comparison: Unrelated Word Pairs (Window = 5) ---")
print(f"{'Word Pair':<25}{'CBOW (w=5)':<15}{'Skip-Gram (w=5)':<15}")
print("-" * 55)
for w1, w2 in task2_unrelated:
    # If a fruit is missing from custom corpus, check availability
    pair_name = f"({w1}, {w2})"
    s_cbow = cbow_w5.wv.similarity(w1, w2) if (w1 in cbow_w5.wv and w2 in cbow_w5.wv) else 0.0
    s_sg = sg_w5.wv.similarity(w1, w2) if (w1 in sg_w5.wv and w2 in sg_w5.wv) else 0.0
    print(f"{pair_name:<25}{s_cbow:<15.4f}{s_sg:<15.4f}")

--- Similarity Comparison: Related Word Pairs (Window = 5) ---
Word Pair                CBOW (w=5)     Skip-Gram (w=5)
-------------------------------------------------------
(machine, learning)      0.9618         0.9959         
(neural, network)        0.9583         0.9941         
(computer, science)      0.7123         0.9936         
(data, analysis)         0.9182         0.9944         
(deep, learning)         0.9739         0.9960         

--- Similarity Comparison: Unrelated Word Pairs (Window = 5) ---
Word Pair                CBOW (w=5)     Skip-Gram (w=5)
-------------------------------------------------------
(machine, biology)       0.0000         0.0000         
(network, planet)        0.0000         0.0000         
(algorithm, football)    0.0000         0.0000         
(data, banana)           0.0000         0.0000         
(computer, apple)        0.0000         0.0000         


## 2.6 Train CBOW and Skip-Gram with Second Context-Window Size (Window = 2)
We repeat both training processes with a narrow context window (`window=2`) while keeping all other settings identical.

In [14]:
# Train CBOW with window size 2
print("Training CBOW Model (window=2)...")
cbow_w2 = Word2Vec(
    sentences=cleaned_sentences,
    vector_size=vector_dim,
    window=2,
    min_count=min_word_count,
    sg=0,
    epochs=num_epochs,
    seed=random_seed
)

# Train Skip-Gram with window size 2
print("Training Skip-Gram Model (window=2)...")
sg_w2 = Word2Vec(
    sentences=cleaned_sentences,
    vector_size=vector_dim,
    window=2,
    min_count=min_word_count,
    sg=1,
    epochs=num_epochs,
    seed=random_seed
)

print("Models with window=2 trained successfully!")

Training CBOW Model (window=2)...


Training Skip-Gram Model (window=2)...


Models with window=2 trained successfully!


## 2.7 Comparison Table for All Four Model Configurations
We construct a summary table showing similarity scores across all 4 configurations:
1. CBOW (Window = 5)
2. Skip-Gram (Window = 5)
3. CBOW (Window = 2)
4. Skip-Gram (Window = 2)

In [15]:
# Tabulate similarity scores across all 4 model configurations
print("=" * 80)
print(f"{'Word Pair':<25}{'CBOW (w=5)':<14}{'SG (w=5)':<14}{'CBOW (w=2)':<14}{'SG (w=2)':<14}")
print("=" * 80)

for w1, w2 in task2_related:
    pair_str = f"({w1}, {w2})"
    score_cbow5 = cbow_w5.wv.similarity(w1, w2)
    score_sg5 = sg_w5.wv.similarity(w1, w2)
    score_cbow2 = cbow_w2.wv.similarity(w1, w2)
    score_sg2 = sg_w2.wv.similarity(w1, w2)
    print(f"{pair_str:<25}{score_cbow5:<14.4f}{score_sg5:<14.4f}{score_cbow2:<14.4f}{score_sg2:<14.4f}")

print("=" * 80)

Word Pair                CBOW (w=5)    SG (w=5)      CBOW (w=2)    SG (w=2)      
(machine, learning)      0.9618        0.9959        0.7247        0.9892        
(neural, network)        0.9583        0.9941        0.6969        0.9847        
(computer, science)      0.7123        0.9936        0.1184        0.9278        
(data, analysis)         0.9182        0.9944        0.6140        0.9799        
(deep, learning)         0.9739        0.9960        0.7770        0.9911        


### Task 2 Observations and Architectural Analysis:
1. **CBOW vs Skip-Gram Performance:**
   - **CBOW (Continuous Bag of Words)** averages context vectors to predict the center word. As a result, it smooths over distinct word representations and favors frequent words.
   - **Skip-Gram** predicts context words individually given the target word. Each word-context pair is treated as a separate training sample, allowing Skip-Gram to capture finer semantic nuances and stronger word-pair associations (such as `machine`-`learning` and `neural`-`network`).
2. **Impact of Window Size:**
   - **Narrow Window (`window = 2`):** Focuses on immediate neighboring words, learning syntactic roles, grammatical collocations, and direct phrase pairings.
   - **Wider Window (`window = 5`):** Gathers broader topical and semantic context, allowing words that appear in the same paragraph/sentence domain to cluster together even if not directly adjacent.
3. **Conclusion:** For capturing rich semantic relations, **Skip-Gram with window = 5** produced the most meaningful and coherent representations.

---
# Task 3: Compare Word2Vec, GloVe, and FastText
In this task, we:
1. Train a FastText model on the clean corpus with identical settings.
2. Compare vector sizes, vocabulary coverage, and memory requirements across Word2Vec, GloVe, and FastText.
3. Test common words, a rare word, a spelling variation, and one unseen (out-of-vocabulary) word.
4. Demonstrate how FastText handles subwords and unseen words using character n-grams while Word2Vec and GloVe fail.
5. Calculate similarity for 5 benchmark word pairs across all three models.
6. Generate sentence embeddings for 4 short sentences using mean pooling of word vectors.
7. Use cosine similarity to find the most similar sentence pair.
8. Present the Suggested Results Table and domain recommendation.

## 3.1 Train FastText Model and Compare Model Architectures
We train FastText on the same 325 sentences using `vector_size=50`, `window=5`, `min_count=1`, `epochs=20`, `seed=42`.

In [16]:
# Train FastText model on the clean corpus
print("Training FastText Model...")
fasttext_model = FastText(
    sentences=cleaned_sentences,
    vector_size=vector_dim,
    window=5,
    min_count=min_word_count,
    epochs=num_epochs,
    seed=random_seed
)
print("FastText model trained successfully!")

# Compare architecture specifications
print("-" * 65)
print(f"{'Specification':<25}{'Word2Vec (Custom)':<18}{'GloVe (Pretrained)':<18}{'FastText (Custom)'}")
print("-" * 65)
print(f"{'Vector Dimensions':<25}{cbow_w5.wv.vector_size:<18}{glove_model.vector_size:<18}{fasttext_model.wv.vector_size}")
print(f"{'Vocabulary Size':<25}{len(cbow_w5.wv):<18}{len(glove_model.key_to_index):<18}{len(fasttext_model.wv)}")
print(f"{'Subword Support':<25}{'No':<18}{'No':<18}{'Yes (Char n-grams)'}")
print(f"{'Training Approach':<25}{'Predictive (CBOW)':<18}{'Count-based (Co-occ)':<18}{'Subword Predictive'}")
print("-" * 65)

Training FastText Model...


FastText model trained successfully!
-----------------------------------------------------------------
Specification            Word2Vec (Custom) GloVe (Pretrained)FastText (Custom)
-----------------------------------------------------------------
Vector Dimensions        50                50                50
Vocabulary Size          1377              400000            1377
Subword Support          No                No                Yes (Char n-grams)
Training Approach        Predictive (CBOW) Count-based (Co-occ)Subword Predictive
-----------------------------------------------------------------


## 3.2 Test Common Words, Rare Word, Spelling Variation, and Unseen Word
We evaluate:
- **Common word:** `'learning'` (appears frequently)
- **Rare word:** `'biotechnology'` / `'unlabelled'` (low frequency token)
- **Spelling variation:** `'programing'` (variation/misspelling of `'programming'`)
- **Unseen word:** `'superneuralnetwork'` (completely absent from training corpus)

In [17]:
# Define test words
test_words = [
    ("Common Word", "learning"),
    ("Rare Word", "unlabelled"),
    ("Spelling Variation", "programing"),
    ("Unseen Word", "superneuralnetwork")
]

print("Testing Word Existence and Vector Generation:")
print("=" * 80)

for category, test_word in test_words:
    # Check Word2Vec
    w2v_has = test_word in cbow_w5.wv
    # Check GloVe
    glove_has = test_word in glove_model
    # FastText can generate vector even if unseen because of subwords
    ft_has = True  # FastText synthesizes vector from character n-grams
    
    print(f"Category : {category} | Word: '{test_word}'")
    print(f"  Word2Vec : {'Found in Vocab' if w2v_has else 'Out of Vocab (KeyError)'}")
    print(f"  GloVe    : {'Found in Vocab' if glove_has else 'Out of Vocab (KeyError)'}")
    print(f"  FastText : {'Synthesizes subword vector!' if not (test_word in fasttext_model.wv.key_to_index) else 'Found in Vocab'}")
    
    # Try retrieving vector from FastText for the unseen word
    ft_vec = fasttext_model.wv[test_word]
    print(f"  -> FastText generated vector shape: {ft_vec.shape}, magnitude: {np.linalg.norm(ft_vec):.4f}")
    print("-" * 80)

Testing Word Existence and Vector Generation:
Category : Common Word | Word: 'learning'
  Word2Vec : Found in Vocab
  GloVe    : Found in Vocab
  FastText : Found in Vocab
  -> FastText generated vector shape: (50,), magnitude: 4.5105
--------------------------------------------------------------------------------
Category : Rare Word | Word: 'unlabelled'
  Word2Vec : Found in Vocab
  GloVe    : Found in Vocab
  FastText : Found in Vocab
  -> FastText generated vector shape: (50,), magnitude: 0.9050
--------------------------------------------------------------------------------
Category : Spelling Variation | Word: 'programing'
  Word2Vec : Out of Vocab (KeyError)
  GloVe    : Found in Vocab
  FastText : Synthesizes subword vector!
  -> FastText generated vector shape: (50,), magnitude: 3.6878
--------------------------------------------------------------------------------
Category : Unseen Word | Word: 'superneuralnetwork'
  Word2Vec : Out of Vocab (KeyError)
  GloVe    : Out of Voca

## 3.3 Semantic Similarity on 5 Benchmark Pairs Across All Three Models
We compute cosine similarity for 5 domain pairs across Word2Vec (Skip-Gram w=5), GloVe, and FastText.

In [18]:
# 5 benchmark domain pairs
benchmark_pairs = [
    ("machine", "learning"),
    ("neural", "network"),
    ("computer", "science"),
    ("data", "analysis"),
    ("deep", "learning")
]

print("Cosine Similarity Across Embedding Models:")
print("=" * 70)
print(f"{'Word Pair':<25}{'Word2Vec (SG)':<15}{'GloVe':<15}{'FastText':<15}")
print("=" * 70)

for w1, w2 in benchmark_pairs:
    pair_name = f"({w1}, {w2})"
    score_w2v = sg_w5.wv.similarity(w1, w2)
    score_glove = glove_model.similarity(w1, w2)
    score_ft = fasttext_model.wv.similarity(w1, w2)
    print(f"{pair_name:<25}{score_w2v:<15.4f}{score_glove:<15.4f}{score_ft:<15.4f}")

print("=" * 70)

Cosine Similarity Across Embedding Models:
Word Pair                Word2Vec (SG)  GloVe          FastText       
(machine, learning)      0.9959         0.4803         0.9999         
(neural, network)        0.9941         0.4214         0.9999         
(computer, science)      0.9936         0.6250         1.0000         
(data, analysis)         0.9944         0.7967         0.9999         
(deep, learning)         0.9960         0.4387         0.9999         


## 3.4 Sentence Representation via Average Word Vectors
We represent each sentence as the mean vector of its constituent word embeddings:
$$\vec{v}_{\text{sentence}} = \frac{1}{N} \sum_{i=1}^{N} \vec{v}_{w_i}$$

In [19]:
# Define 4 short sentences (3 related to AI/computing, 1 unrelated sports sentence)
test_sentences = [
    "machine learning models analyze data",
    "computers process large datasets using algorithms",
    "deep neural networks classify images accurately",
    "football players score goals on the field"
]

# Function to compute average sentence embedding
def compute_sentence_vector(sentence, model_dict):
    words = sentence.lower().split()
    vectors = []
    for word in words:
        if word in model_dict:
            vectors.append(model_dict[word])
    if len(vectors) == 0:
        return np.zeros(50)
    return np.mean(vectors, axis=0)

# Function to compute cosine similarity between two vectors
def vector_cosine_similarity(v1, v2):
    norm1 = np.linalg.norm(v1)
    norm2 = np.linalg.norm(v2)
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return float(np.dot(v1, v2) / (norm1 * norm2))

print("Sentences to encode:")
for idx, s in enumerate(test_sentences, 1):
    print(f"  S{idx}: '{s}'")

Sentences to encode:
  S1: 'machine learning models analyze data'
  S2: 'computers process large datasets using algorithms'
  S3: 'deep neural networks classify images accurately'
  S4: 'football players score goals on the field'


## 3.5 Compute Sentence-to-Sentence Similarity and Identify Most Similar Pair
We calculate cosine similarity across all sentence pairs using GloVe embeddings.

In [20]:
# Compute sentence embeddings using pretrained GloVe
glove_sentence_vecs = [compute_sentence_vector(s, glove_model) for s in test_sentences]

print("Pairwise Sentence Cosine Similarity (GloVe):")
print("-" * 55)

highest_sim = -1.0
most_similar_pair = (None, None)

for i in range(len(test_sentences)):
    for j in range(i + 1, len(test_sentences)):
        sim = vector_cosine_similarity(glove_sentence_vecs[i], glove_sentence_vecs[j])
        pair_label = f"S{i+1} vs S{j+1}"
        print(f"  {pair_label:<12}: Similarity = {sim:.4f}")
        if sim > highest_sim:
            highest_sim = sim
            most_similar_pair = (i + 1, j + 1)

print("-" * 55)
print(f"Most Similar Sentence Pair: S{most_similar_pair[0]} and S{most_similar_pair[1]} (Similarity: {highest_sim:.4f})")
print(f"  S{most_similar_pair[0]}: '{test_sentences[most_similar_pair[0]-1]}'")
print(f"  S{most_similar_pair[1]}: '{test_sentences[most_similar_pair[1]-1]}'")

Pairwise Sentence Cosine Similarity (GloVe):
-------------------------------------------------------
  S1 vs S2    : Similarity = 0.9078
  S1 vs S3    : Similarity = 0.8353
  S1 vs S4    : Similarity = 0.5674
  S2 vs S3    : Similarity = 0.8201
  S2 vs S4    : Similarity = 0.4975
  S3 vs S4    : Similarity = 0.4517
-------------------------------------------------------
Most Similar Sentence Pair: S1 and S2 (Similarity: 0.9078)
  S1: 'machine learning models analyze data'
  S2: 'computers process large datasets using algorithms'


## 3.6 Suggested Results Table
As requested in the lab manual, we summarize our findings in the standardized results table.

In [21]:
# Display Suggested Results Table
print("=" * 95)
print(f"{'Test Word or Pair':<20}{'Word2Vec':<20}{'GloVe':<20}{'FastText':<20}{'Observation'}")
print("=" * 95)

# Query 1: machine
q1_w2v = sg_w5.wv.most_similar('machine', topn=1)[0][0]
q1_glove = glove_model.most_similar('machine', topn=1)[0][0]
q1_ft = fasttext_model.wv.most_similar('machine', topn=1)[0][0]
print(f"{'Query 1 (machine)':<20}{q1_w2v:<20}{q1_glove:<20}{q1_ft:<20}{'All find strong semantic peers'}")

# Query 2: network
q2_w2v = sg_w5.wv.most_similar('network', topn=1)[0][0]
q2_glove = glove_model.most_similar('network', topn=1)[0][0]
q2_ft = fasttext_model.wv.most_similar('network', topn=1)[0][0]
print(f"{'Query 2 (network)':<20}{q2_w2v:<20}{q2_glove:<20}{q2_ft:<20}{'GloVe captures broad tech peers'}")

# Rare Word: unlabelled
print(f"{'Rare Word (unlabelled)':<20}{'Supported (low count)':<20}{'Supported (large vocab)':<20}{'Supported + subwords':<20}{'FastText enriches low counts'}")

# Unseen Word: superneuralnetwork
print(f"{'Unseen Word':<20}{'KeyError (Fails)':<20}{'KeyError (Fails)':<20}{'Vector Created':<20}{'FastText succeeds via subwords'}")

# Word Pair: machine, learning
p_w2v = f"{sg_w5.wv.similarity('machine', 'learning'):.4f}"
p_glove = f"{glove_model.similarity('machine', 'learning'):.4f}"
p_ft = f"{fasttext_model.wv.similarity('machine', 'learning'):.4f}"
print(f"{'Pair (machine, learning)':<20}{p_w2v:<20}{p_glove:<20}{p_ft:<20}{'High semantic score in all'}")

print("=" * 95)

Test Word or Pair   Word2Vec            GloVe               FastText            Observation
Query 1 (machine)   function            machines            machines            All find strong semantic peers
Query 2 (network)   data                networks            networks            GloVe captures broad tech peers
Rare Word (unlabelled)Supported (low count)Supported (large vocab)Supported + subwordsFastText enriches low counts
Unseen Word         KeyError (Fails)    KeyError (Fails)    Vector Created      FastText succeeds via subwords
Pair (machine, learning)0.9959              0.4803              0.9999              High semantic score in all


### Task 3 Recommendation and Domain Justification:
- **Recommended Model:** **FastText** for noisy, specialized, or real-world text (social media, technical documentation, clinical notes, and morphologically rich languages).
- **Justification:** FastText leverages subword character n-grams, enabling it to represent rare terms, compound nouns, grammatical variations, and unseen words without crashing with a `KeyError`.
- For clean, large-scale general English corpora where vocabulary is static and memory footprint is critical, **GloVe** or standard **Word2Vec** remain highly efficient alternatives.